In [ ]:
import os
os.environ["ROGII_GOLD_PROFILE"] = "balanced"  # conservative | balanced | aggressive

import sys, os, glob, subprocess
# koolbox setup: wheel install or sys.path fallback
kb_dir = '/kaggle/input/koolbox-offline'
if not os.path.isdir(kb_dir):
    cand = glob.glob('/kaggle/input/**/koolbox*', recursive=True)
    print('koolbox candidates:', cand[:5])
    if cand: kb_dir = cand[0]
print('using koolbox dir:', kb_dir)
if os.path.isdir(kb_dir):
    print('listing:', os.listdir(kb_dir)[:20])
    whls = glob.glob(f'{kb_dir}/**/*.whl', recursive=True)
    if whls:
        for w in whls:
            print('install', w)
            subprocess.run(['pip', 'install', '--no-deps', w], check=False)
    else:
        sys.path.insert(0, kb_dir)
        for sub in os.listdir(kb_dir):
            sub_path = os.path.join(kb_dir, sub)
            if os.path.isdir(sub_path):
                sys.path.insert(0, sub_path)
import koolbox
print('koolbox OK:', koolbox.__file__)


In [ ]:
from lightgbm import LGBMRegressor, log_evaluation, early_stopping
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import Ridge, ElasticNet
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from joblib import Parallel, delayed
from koolbox import Trainer
from pathlib import Path
from numba import njit
import matplotlib.pyplot as plt
import multiprocessing
import pandas as pd
import numpy as np
import warnings
import joblib
import time
import gc

warnings.filterwarnings("ignore")
print('All imports OK')


In [ ]:
class CFG:
    dataset_path = Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction")
    artifacts_path = Path("/kaggle/input/datasets/ravaghi/wellbore-geology-prediction-artifacts")
    
    seed = 42
    n_splits = 5
    cv = GroupKFold(n_splits=n_splits)
    
    metric = root_mean_squared_error
    USE_GPU = True  # set False for CPU-only
    SHOW_FIGS = False


In [ ]:
SEED=42
NCPU=min(4,multiprocessing.cpu_count())

SELECTOR_N_EVAL_THRESHOLD = 4840.0
SELECTOR_Z_SPAN_THRESHOLDS  = (136.73, 185.51)

SELECTOR_BIN_VARIANTS = {
    0: 'pf_scale_5_hold_0.2',
    1: 'pf_scale_3_hold_0.15',
    2: 'pf_scale_12_beam_0.2_hold_0.15',
    3: 'pf_scale_5_hold_0.15',
    4: 'pf_scale_5_beam_0.05_hold_0.05',
    5: 'pf_scale_12_beam_0.2_hold_0.05',
}

SELECTOR_GLOBAL_VARIANT = 'pf_scale_8_hold_0.2'
SELECTOR_SCALES = (3.0, 5.0, 8.0, 12.0)

FORMATIONS = ["ANCC","ASTNU","ASTNL","EGFDU","EGFDL","BUDA"]
PLANE_K=10; DENSE_SPW=60; DENSE_K=20; N_SPLITS=5

# === Super6: 15 beam configs (was 7 in baseline) ===
BEAMS=[
    (10,20.0,144.0,2,"cons"),
    (10, 8.0, 64.0,2,"loose"),
    ( 8,35.0,220.0,1,"vcons"),
    (10,14.0, 90.0,5,"sm5"),
    (20, 4.0, 36.0,3,"vloose"),
    (12,12.0,100.0,3,"mid"),
    (15,25.0,180.0,2,"stiff"),
    (20,30.0,200.0,2,"stiff2"),
    (15,10.0, 80.0,4,"loose2"),
    (25, 6.0, 50.0,3,"vloose2"),
    (10,40.0,300.0,1,"vcons2"),
    (12,18.0,120.0,5,"sm5b"),
    (30, 8.0, 70.0,2,"beam13"),
    (10,50.0,400.0,0,"beam14"),
    ( 8,15.0,100.0,3,"beam15"),
]
BEAM_CONFIGS = [(bs, mc, es, r) for (bs, mc, es, r, _) in BEAMS]

PF_N=800; ANCC_N=800
PF_MOM=0.993; PF_VN=0.004; PF_PN=0.008
PF_GR_SIG_MIN=8.; PF_GR_SIG_MAX=65.; PF_GR_SIG_DEF=25.
PF_INIT_V_STD=0.02; PF_RESAMP=0.45
PF_ROUGH_P=0.18; PF_ROUGH_V=0.0025; PF_GR_WIN=7; PF_GR_WT=0.35
ANCC_ALPHA=0.998; ANCC_RN=0.002; ANCC_PN=0.005
ANCC_IR=0.01; ANCC_IS=0.35; ANCC_RP=0.12; ANCC_RR=0.001

print(f'{len(BEAMS)} beams | PF N={PF_N} ANCC_N={ANCC_N}')


In [ ]:
@njit(cache=True)
def _interp1(grid, v, vmin, step):
    i = int((v - vmin) / step)
    if i < 0: return grid[0]
    n = len(grid) - 1
    if i >= n: return grid[n]
    t = (v - vmin) / step - i
    return grid[i]*(1.-t) + grid[i+1]*t

@njit(cache=True)
def _resamp(pos, aux, w, N, rp, rv):
    cum = np.zeros(N+1)
    for j in range(N): cum[j+1]=cum[j]+w[j]
    u0=np.random.uniform(0.,1./N); np2=np.empty(N); na=np.empty(N); ci=0
    for j in range(N):
        u=u0+j/N
        while ci<N-1 and cum[ci+1]<u: ci+=1
        np2[j]=pos[ci]+rp*np.random.randn()
        na[j] =aux[ci]+rv*np.random.randn()
    return np2,na

@njit(cache=True)
def _beam_jit(sgr, tw_gr, si, BS, mc, es):
    n=len(sgr); nt=len(tw_gr); MAX=BS*6
    bidx=np.zeros(BS,np.int64); bidx[0]=si
    bcost=np.full(BS,1e30);     bcost[0]=0.; bn=np.int64(1)
    hI=np.zeros((n,BS),np.int64); hP=np.zeros((n,BS),np.int64)
    cI=np.zeros(MAX,np.int64); cC=np.full(MAX,1e30); cP=np.zeros(MAX,np.int64)
    for step in range(n):
        gv=sgr[step]; nc=np.int64(0)
        for bi in range(bn):
            idx=bidx[bi]; cost=bcost[bi]
            for d in range(-2,3):
                ni=idx+d
                if ni<0 or ni>=nt: continue
                tot=cost+(gv-tw_gr[ni])**2/es+mc*(d if d>=0 else -d)
                fnd=np.int64(-1)
                for ci in range(nc):
                    if cI[ci]==ni: fnd=ci; break
                if fnd>=0:
                    if tot<cC[fnd]: cC[fnd]=tot; cP[fnd]=bi
                else:
                    if nc<MAX: cI[nc]=ni; cC[nc]=tot; cP[nc]=bi; nc+=1
        kept=min(BS,nc)
        for i in range(kept):
            mi=i
            for j in range(i+1,nc):
                if cC[j]<cC[mi]: mi=j
            if mi!=i:
                cI[i],cI[mi]=cI[mi],cI[i]
                cC[i],cC[mi]=cC[mi],cC[i]
                cP[i],cP[mi]=cP[mi],cP[i]
        hI[step,:kept]=cI[:kept]; hP[step,:kept]=cP[:kept]
        bidx[:kept]=cI[:kept]; bcost[:kept]=cC[:kept]; bn=kept
    best=np.int64(0)
    for b in range(1,bn):
        if bcost[b]<bcost[best]: best=b
    path=np.zeros(n,np.int64); b=best
    for s in range(n-1,-1,-1): path[s]=hI[s,b]; b=hP[s,b]
    return path

@njit(cache=True)
def _pf_ancc(md_v,z_v,gr_v,gg,vmin,step,gs,ls,ir,N,
              ALPHA,RN,PN,IS,RP,RR,RESAMP):
    pos=np.empty(N); rate=np.empty(N); w=np.ones(N)/N
    for j in range(N):
        pos[j]=ls+IS*np.random.randn()
        rate[j]=ir+0.01*np.random.randn()
    pts=np.empty(len(md_v)); std_=np.empty(len(md_v)); pm=md_v[0]-1.
    for i in range(len(md_v)):
        dm=md_v[i]-pm; dm=max(dm,1.)
        for j in range(N):
            rate[j]=ALPHA*rate[j]+RN*np.random.randn()
            pos[j]+=rate[j]*dm+PN*np.random.randn()
            tvt_j=pos[j]-z_v[i]
            tvt_j=max(tvt_j,vmin-50.); tvt_j=min(tvt_j,vmin+len(gg)*step+50.)
            pos[j]=tvt_j+z_v[i]
        if not np.isnan(gr_v[i]):
            ws=0.
            for j in range(N):
                eg=_interp1(gg,pos[j]-z_v[i],vmin,step)
                d=(gr_v[i]-eg)/gs
                lk=max(np.exp(-0.5*d*d) if d*d<600. else 0.,1e-300)
                w[j]*=lk; ws+=w[j]
            if ws>0.:
                for j in range(N): w[j]/=ws
            else:
                for j in range(N): w[j]=1./N
        ne=0.
        for j in range(N): ne+=w[j]*w[j]
        if 1./ne<RESAMP*N:
            pos,rate=_resamp(pos,rate,w,N,RP,RR)
            for j in range(N): w[j]=1./N
        tv=0.
        for j in range(N): tv+=w[j]*(pos[j]-z_v[i])
        pts[i]=tv; va=0.
        for j in range(N): va+=w[j]*(pos[j]-z_v[i]-tv)**2
        std_[i]=va**0.5; pm=md_v[i]
    return pts,std_

@njit(cache=True)
def _pf_z(md_v,z_v,gr_v,gr_sm_v,gg_p,gg_s,vmin,step,
          gs,ip,iv,beta,icpt,zsig,N,
          MOM,VN,PN,GR_WT,RP,RV,RESAMP):
    pos=np.empty(N); vel=np.empty(N); w=np.ones(N)/N
    for j in range(N):
        pos[j]=ip+0.5*np.random.randn()
        vel[j]=iv+0.02*np.random.randn()
    pts=np.empty(len(md_v)); std_=np.empty(len(md_v)); pm=md_v[0]-1.; pz=z_v[0]-1.
    for i in range(len(md_v)):
        dm=md_v[i]-pm; dm=max(dm,1.)
        dzd=(z_v[i]-pz)/dm; ve=beta*dzd+icpt
        for j in range(N):
            vel[j]=MOM*vel[j]+VN*np.random.randn()
            pos[j]+=vel[j]*dm+PN*np.random.randn()
            pos[j]=max(pos[j],vmin-50.); pos[j]=min(pos[j],vmin+len(gg_p)*step+50.)
        if not np.isnan(gr_v[i]):
            ws=0.
            for j in range(N):
                ep=_interp1(gg_p,pos[j],vmin,step)
                dp=(gr_v[i]-ep)/gs
                lp=max(np.exp(-0.5*dp*dp) if dp*dp<600. else 0.,1e-300)
                if not np.isnan(gr_sm_v[i]):
                    es=_interp1(gg_s,pos[j],vmin,step)
                    ds=(gr_sm_v[i]-es)/(gs*1.5)
                    ls=max(np.exp(-0.5*ds*ds) if ds*ds<600. else 0.,1e-300)
                    lk=(1.-GR_WT)*lp+GR_WT*ls
                else: lk=lp
                lk=max(lk,1e-300); w[j]*=lk; ws+=w[j]
            if ws>0.:
                for j in range(N): w[j]/=ws
            else:
                for j in range(N): w[j]=1./N
        ws2=0.
        for j in range(N):
            dv=(vel[j]-ve)/max(zsig*2.,0.005)
            lz=max(np.exp(-0.5*dv*dv) if dv*dv<600. else 0.,1e-300)
            w[j]*=lz; ws2+=w[j]
        if ws2>0.:
            for j in range(N): w[j]/=ws2
        else:
            for j in range(N): w[j]=1./N
        ne=0.
        for j in range(N): ne+=w[j]*w[j]
        if 1./ne<RESAMP*N:
            pos,vel=_resamp(pos,vel,w,N,RP,RV)
            for j in range(N): w[j]=1./N
        wm=0.
        for j in range(N): wm+=w[j]*pos[j]
        pts[i]=wm; va=0.
        for j in range(N): va+=w[j]*(pos[j]-wm)**2
        std_[i]=va**0.5; pm=md_v[i]; pz=z_v[i]
    return pts,std_

# Warm up all JIT kernels
_md=np.linspace(1,50,20,np.float64); _z=np.zeros(20,np.float64); _gr=np.full(20,50.,np.float64)
_gg=np.linspace(45,55,100,np.float64)
_pf_ancc(_md,_z,_gr,_gg,45.,0.1,20.,50.,0.,8,0.998,0.002,0.005,0.3,0.1,0.001,0.5)
_pf_z(_md,_z,_gr,_gr,_gg,_gg,45.,0.1,20.,50.,0.,-1.,0.,0.1,8,0.993,0.005,0.01,0.3,0.2,0.003,0.5)
_beam_jit(np.random.randn(30),np.random.randn(50),25,8,15.,100.)
print('JIT warmup complete')


In [ ]:
# Feature engineering helpers ==============================================

def _grid(tw_tvt,tw_gr,step=0.2):
    tmin=float(tw_tvt.min()); tmax=float(tw_tvt.max())
    tvt_g=np.arange(tmin,tmax+step,step)
    return np.interp(tvt_g,tw_tvt,tw_gr).astype(np.float64),float(tmin),float(step)

def _gr_sig(hw,tw_tvt,tw_gr):
    kn=hw[hw['TVT_input'].notna()&hw['GR'].notna()]
    if len(kn)<20: return float(PF_GR_SIG_DEF)
    return float(np.clip(np.std(kn['GR'].values-np.interp(kn['TVT_input'].values,tw_tvt,tw_gr)),
                          PF_GR_SIG_MIN,PF_GR_SIG_MAX))

def _nn(arr,v):
    i=int(np.searchsorted(arr,v,'left'))
    if i>=len(arr): return len(arr)-1
    if i>0 and abs(arr[i-1]-v)<=abs(arr[i]-v): return i-1
    return i

def _smooth(vals,fb,r):
    s=pd.Series(vals,dtype='float32').interpolate(limit_direction='both').fillna(fb)
    return (s.rolling(r*2+1,center=True,min_periods=1).mean() if r>0 else s).to_numpy(np.float32)

def robust_slope(x,y):
    x=np.asarray(x,float); y=np.asarray(y,float)
    m=np.isfinite(x)&np.isfinite(y)
    if m.sum()<2 or np.std(x[m])<1e-6: return 0.
    return float(np.polyfit(x[m],y[m],1)[0])

def affine_cal(kgr,tw_at_k,min_pts=20):
    v=np.isfinite(kgr)&np.isfinite(tw_at_k)
    if v.sum()<min_pts or np.std(tw_at_k[v])<1e-6:
        return 1.,float(np.nanmean(kgr)-np.nanmean(tw_at_k)) if v.any() else 0.
    a,b=np.polyfit(tw_at_k[v],kgr[v],1); return float(a),float(b)

def seg_b_well(ktvt,kz,form_col):
    bv=ktvt+kz-form_col; n=len(bv)
    b_full=float(np.median(bv))
    b_late=float(np.median(bv[max(0,n-50):])) if n>=5 else b_full
    t1,t2=n//3, 2*n//3
    b_early=float(np.median(bv[:max(1,t1)])) if t1>0 else b_full
    b_mid  =float(np.median(bv[t1:max(t1+1,t2)])) if t2>t1 else b_full
    w=np.exp(0.02*np.arange(n)); w/=w.sum()
    b_wls=float(np.dot(w,bv))
    xx=np.arange(n,dtype=float); mask=np.isfinite(bv)
    if mask.sum()>4:
        b_trend=float(np.polyval(np.polyfit(xx[mask],bv[mask],1),n-1))
    else: b_trend=b_full
    return b_full,b_early,b_mid,b_late,b_wls,b_trend

def beam_search(gr_h,tw_tvt,tw_gr,start_tvt,bs,mc,es,r):
    si=_nn(tw_tvt,start_tvt)
    if r>0 and len(gr_h)>max(3,2*r+1):
        sgr=savgol_filter(gr_h,min(2*r+1,len(gr_h)-1 if len(gr_h)%2==1 else len(gr_h)-2),
                          min(2,min(2*r+1,len(gr_h))-1))
    else: sgr=gr_h.copy()
    sgr=_smooth(sgr,float(np.nanmean(tw_gr)),0).astype(np.float64)
    path=_beam_jit(sgr,tw_gr.astype(np.float64),si,bs,float(mc),float(es))
    return tw_tvt[path].astype(np.float32)

def multi_scale_ncc(kgr,ktvt,hgr,hws=(8,15,25,50),stride=3):
    out=[]
    for hw in hws:
        win=2*hw+1; nk=len(kgr); nh=len(hgr)
        if nk<win+1 or nh==0:
            out.append((np.full(nh,ktvt[-1],np.float32),np.zeros(nh,np.float32))); continue
        kg=pd.Series(kgr).rolling(5,center=True,min_periods=1).mean().values.astype(np.float32)
        hg=pd.Series(hgr).rolling(5,center=True,min_periods=1).mean().values.astype(np.float32)
        sts=np.arange(0,nk-win+1,stride,dtype=np.int32); M=len(sts)
        if M==0:
            out.append((np.full(nh,ktvt[-1],np.float32),np.zeros(nh,np.float32))); continue
        C=kg[sts[:,None]+np.arange(win,dtype=np.int32)[None,:]].astype(np.float32)
        Cn=(C-C.mean(1,keepdims=True))/(C.std(1,keepdims=True)+1e-6)
        hp=np.pad(hg,hw,mode='edge')
        H=hp[np.arange(nh)[:,None]+np.arange(win)[None,:]].astype(np.float32)
        Hn=(H-H.mean(1,keepdims=True))/(H.std(1,keepdims=True)+1e-6)
        ncc=Hn@Cn.T/win; best=ncc.argmax(1); score=ncc.max(1).astype(np.float32)
        out.append((ktvt[np.clip(sts[best]+hw,0,nk-1)].astype(np.float32),score))
    tvts=np.stack([o[0] for o in out],1); scores=np.stack([o[1] for o in out],1)
    sw=np.exp(3.*scores); sw/=sw.sum(1,keepdims=True)+1e-9
    sc_ens=(tvts*sw).sum(1).astype(np.float32)
    return out, sc_ens

print('Feature helpers OK')


In [ ]:
def run_pf_ancc(hw,tw_tvt,tw_gr,N=ANCC_N):
    gs=_gr_sig(hw,tw_tvt,tw_gr)
    kn=hw[hw['TVT_input'].notna()]; ev=hw[hw['TVT_input'].isna()]
    if len(ev)==0: return np.array([]),np.array([])
    ls=float(kn['TVT_input'].iloc[-1]+kn['Z'].iloc[-1])
    tail=kn.tail(30); dt=np.diff(tail['TVT_input'].values)
    dz=np.diff(tail['Z'].values); dm=np.diff(tail['MD'].values); m=dm>0
    ir=float(np.median((dt+dz)[m]/dm[m])) if m.sum()>=3 else 0.
    gg,gmin,gst=_grid(tw_tvt,tw_gr)
    pts,std=_pf_ancc(ev['MD'].values.astype(np.float64),ev['Z'].values.astype(np.float64),
                      ev['GR'].values.astype(np.float64),gg,gmin,gst,
                      gs,ls,ir,N,ANCC_ALPHA,ANCC_RN,ANCC_PN,ANCC_IS,ANCC_RP,ANCC_RR,PF_RESAMP)
    return pts.astype(np.float32),std.astype(np.float32)

def run_pf_z(hw,tw_tvt,tw_gr,N=PF_N):
    gs=_gr_sig(hw,tw_tvt,tw_gr)
    tw_s=pd.Series(tw_gr).rolling(PF_GR_WIN,center=True,min_periods=1).mean().values.astype(np.float32)
    kna=hw[hw['TVT_input'].notna()]; ev=hw[hw['TVT_input'].isna()]
    if len(ev)==0: return np.array([]),np.array([])
    dz_k=np.diff(kna['Z'].values); dvt=np.diff(kna['TVT_input'].values)
    dmd_k=np.diff(kna['MD'].values); m2=dmd_k>0
    if m2.sum()>=10:
        vz=dz_k[m2]/dmd_k[m2]; vt=dvt[m2]/dmd_k[m2]
        A=np.column_stack([vz,np.ones_like(vz)]); c,_,_,_=np.linalg.lstsq(A,vt,rcond=None)
        beta,icpt,zsig=float(c[0]),float(c[1]),max(float(np.std(vt-(c[0]*vz+c[1]))),0.001)
    else: beta,icpt,zsig=-1.,0.,0.1
    t2=kna.tail(20); dvt2=np.diff(t2['TVT_input'].values); dmd2=np.diff(t2['MD'].values); m3=dmd2>0
    iv=float(np.median(dvt2[m3]/dmd2[m3])) if m3.sum()>=3 else 0.
    gg,gmin,gst=_grid(tw_tvt,tw_gr)
    gs2,_,_=_grid(tw_tvt,tw_s)
    gr_sm=hw['GR'].rolling(PF_GR_WIN,center=True,min_periods=1).mean()
    pts,std=_pf_z(ev['MD'].values.astype(np.float64),ev['Z'].values.astype(np.float64),
                   ev['GR'].values.astype(np.float64),
                   gr_sm.loc[ev.index].values.astype(np.float64),
                   gg,gs2,gmin,gst,gs,float(kna['TVT_input'].iloc[-1]),iv,
                   beta,icpt,zsig,N,
                   PF_MOM,PF_VN,PF_PN,PF_GR_WT,PF_ROUGH_P,PF_ROUGH_V,PF_RESAMP)
    return pts.astype(np.float32),std.astype(np.float32)

# Likelihood-weighted PF ensemble (for 128-seed slow PF on test)
def run_pf_lik_ensemble_scales(hw, tw, scales=SELECTOR_SCALES, n_particles=500, n_seeds=128):
    preds=[]; liks=[]
    for s in range(n_seeds):
        p, ll = run_particle_filter(hw, tw, n_particles=n_particles, seed=s)
        preds.append(p); liks.append(ll)
    pred_arr=np.stack(preds,0); liks=np.array(liks); liks_n=liks-liks.max()
    out={}
    for scale in scales:
        w=np.exp(liks_n/float(scale)); w/=w.sum()
        out[f'pf_scale_{scale:g}']=(w[:,None]*pred_arr).sum(0)
    out['pf_mean']=pred_arr.mean(0)
    return out

def run_particle_filter(hw, tw, n_particles=500, seed=42):
    tw_s=tw.sort_values('TVT'); tw_tvt=tw_s['TVT'].values.astype(float)
    tw_gr=tw_s['GR'].fillna(tw_s['GR'].mean()).values.astype(float)
    kn=hw[hw['TVT_input'].notna()]; ev=hw[hw['TVT_input'].isna()]
    if len(ev)==0: return hw['TVT_input'].values.astype(float).copy(),0.0
    last=kn.iloc[-1]
    last_tvt=float(last['TVT_input']); last_Z=float(last['Z']); last_MD=float(last['MD'])
    tw_at_k=np.interp(kn['TVT_input'].values,tw_tvt,tw_gr)
    gs=float(np.clip(np.nanstd(kn['GR'].fillna(0).values-tw_at_k),10.,60.))
    tail=kn.tail(30)
    dt=np.diff(tail['TVT_input'].values); dz=np.diff(tail['Z'].values); dm=np.diff(tail['MD'].values); m=dm>0
    ir=float(np.median((dt+dz)[m]/dm[m])) if m.sum()>=3 else 0.0
    N=n_particles; rng=np.random.default_rng(seed)
    ls=last_tvt+last_Z; pos=ls+4.5*rng.standard_normal(N)
    rate=ir+0.01*rng.standard_normal(N); w=np.ones(N)/N
    MOM=0.998; VN=0.002; PN=0.005; RP=0.1; RR=0.001; RESAMP=0.5
    md_v=ev['MD'].values.astype(float); z_v=ev['Z'].values.astype(float)
    gr_interp=hw['GR'].interpolate(limit_direction='both').fillna(tw_gr.mean())
    gr_v=gr_interp.values.astype(float)[ev.index]
    out_vals=hw['TVT_input'].values.astype(float).copy(); res=np.empty(len(ev))
    prev_MD=last_MD; log_lik=0.0
    for i in range(len(ev)):
        dm_step=max(md_v[i]-prev_MD,1.0)
        rate=MOM*rate+VN*rng.standard_normal(N)
        pos=pos+rate*dm_step+PN*rng.standard_normal(N)
        tvt_p=pos-z_v[i]
        tvt_p=np.clip(tvt_p,tw_tvt[0]-100,tw_tvt[-1]+100); pos=tvt_p+z_v[i]
        eg=np.interp(tvt_p,tw_tvt,tw_gr)
        d=(gr_v[i]-eg)/gs; lk=np.exp(-0.5*np.minimum(d**2,600.))
        lk=np.maximum(lk,1e-300); avg_lk=float((w*lk).sum()); log_lik+=np.log(max(avg_lk,1e-300))
        w=w*lk; ws=w.sum(); w=w/ws if ws>0 else np.ones(N)/N
        n_eff=1.0/(w**2).sum()
        if n_eff<RESAMP*N:
            cum=np.cumsum(w); u0=rng.uniform(0,1.0/N)
            idx=np.clip(np.searchsorted(cum,u0+np.arange(N)/N),0,N-1)
            pos=pos[idx]+RP*rng.standard_normal(N)
            rate=rate[idx]+RR*rng.standard_normal(N); w=np.ones(N)/N
        res[i]=float(np.dot(w,pos-z_v[i])); prev_MD=md_v[i]
    out_vals[list(ev.index)]=res; return out_vals,log_lik

def run_beam_ensemble(hw,tw):
    kn=hw[hw['TVT_input'].notna()]; ev=hw[hw['TVT_input'].isna()]
    if len(ev)==0: return hw['TVT_input'].values.astype(float).copy()
    last_tvt=float(kn.iloc[-1]['TVT_input'])
    tw_s=tw.sort_values('TVT'); tw_tvt=tw_s['TVT'].values.astype(float)
    tw_gr=tw_s['GR'].fillna(tw_s['GR'].mean()).values.astype(float)
    gr_all=hw['GR'].interpolate(limit_direction='both').fillna(tw_gr.mean()).values.astype(float)
    hgr=gr_all[ev.index]
    beam_results=[beam_search(hgr,tw_tvt,tw_gr,last_tvt,bs,mc,es,r) for (bs,mc,es,r) in BEAM_CONFIGS]
    beam_mean=np.stack(beam_results,0).mean(0)
    out=hw['TVT_input'].values.astype(float).copy(); out[list(ev.index)]=beam_mean; return out

def selector_well_code(hw):
    eval_mask=hw['TVT_input'].isna().to_numpy()
    n_eval=float(eval_mask.sum())
    z_eval=hw.loc[eval_mask,'Z'].values.astype(float)
    z_span=float(np.nanmax(z_eval)-np.nanmin(z_eval)) if len(z_eval) else 0.0
    n_bin=int(n_eval>SELECTOR_N_EVAL_THRESHOLD)
    z_bin=int(np.searchsorted(SELECTOR_Z_SPAN_THRESHOLDS,z_span,side='right'))
    code=n_bin+2*z_bin
    variant=SELECTOR_BIN_VARIANTS.get(code,SELECTOR_GLOBAL_VARIANT)
    return code,variant,n_eval,z_span

def parse_selector_variant(name):
    parts=name.split('_'); scale=float(parts[2])
    beam_weight=0.0; hold_weight=0.0
    if 'beam' in parts: beam_weight=float(parts[parts.index('beam')+1])
    if 'hold' in parts: hold_weight=float(parts[parts.index('hold')+1])
    return scale,beam_weight,hold_weight

def apply_selector_variant(name,pf_by_scale,tvt_beam,last_known_tvt):
    scale,beam_weight,hold_weight=parse_selector_variant(name)
    base=pf_by_scale.get(f'pf_scale_{scale:g}')
    if base is None: base=pf_by_scale.get('pf_scale_8')
    if base is None: base=tvt_beam
    pred=(1.0-beam_weight)*base+beam_weight*tvt_beam
    pred=(1.0-hold_weight)*pred+hold_weight*last_known_tvt; return pred

print('PF runners OK')


In [ ]:
class FormationPlaneKNN:
    def __init__(self,well_ids,data_dir):
        rows=[]
        for wid in well_ids:
            p=data_dir/f'{wid}__horizontal_well.csv'
            try: df=pd.read_csv(p,usecols=['X','Y']+FORMATIONS).dropna()
            except: continue
            if len(df)==0: continue
            row={'wid':wid,'x':float(df['X'].median()),'y':float(df['Y'].median())}
            for c in FORMATIONS: row[f'{c}_m']=float(df[c].median())
            rows.append(row)
        self.df=pd.DataFrame(rows); self.wmap={w:i for i,w in enumerate(self.df['wid'])}
        xy=self.df[['x','y']].to_numpy(); self.scale=np.where(xy.std(0)<1e-3,1.,xy.std(0))
        self.tree=cKDTree(xy/self.scale)
        self.xa=self.df['x'].to_numpy(); self.ya=self.df['y'].to_numpy()
        self.fa=self.df[[f'{c}_m' for c in FORMATIONS]].to_numpy(np.float64)

    def impute(self,xy_q,self_wid=None,k=PLANE_K):
        q=xy_q/self.scale; nf=min(k+5,len(self.df))
        dist,idx=self.tree.query(q,k=nf,workers=-1)
        if self_wid in self.wmap: dist=np.where(idx==self.wmap[self_wid],np.inf,dist)
        ord=np.argpartition(dist,min(k-1,nf-1),1)[:,:k]
        dk=np.take_along_axis(dist,ord,1); ik=np.take_along_axis(idx,ord,1)
        vk=np.isfinite(dk); w=np.where(vk,1./(dk+1e-3),0.).astype(np.float64)
        xn=self.xa[ik]; yn=self.ya[ik]; fn=self.fa[ik]; wx=w*xn; wy=w*yn
        A=np.zeros((len(q),3,3))
        A[:,0,0]=(wx*xn).sum(1); A[:,0,1]=(wx*yn).sum(1); A[:,0,2]=wx.sum(1)
        A[:,1,0]=A[:,0,1]; A[:,1,1]=(wy*yn).sum(1); A[:,1,2]=wy.sum(1)
        A[:,2,0]=A[:,0,2]; A[:,2,1]=A[:,1,2]; A[:,2,2]=w.sum(1)
        A[:,0,0]+=1e-9; A[:,1,1]+=1e-9; A[:,2,2]+=1e-9
        rhs=np.stack([(wx[:,:,None]*fn).sum(1),(wy[:,:,None]*fn).sum(1),(w[:,:,None]*fn).sum(1)],1)
        try: coef=np.linalg.solve(A,rhs)
        except:
            coef=np.zeros((len(q),3,6))
            for r in range(len(q)):
                try: coef[r]=np.linalg.pinv(A[r])@rhs[r]
                except: pass
        Xq=xy_q[:,0]; Yq=xy_q[:,1]
        pred=(Xq[:,None]*coef[:,0,:]+Yq[:,None]*coef[:,1,:]+coef[:,2,:]).astype(np.float32)
        pred[~vk.any(1)]=self.fa.mean(0)
        return pred,np.where(vk,dk,np.inf).min(1).astype(np.float32)

class DenseANCCImputer:
    def __init__(self,well_ids,data_dir,spw=DENSE_SPW):
        xs,ys,anccs,wids=[],[],[],[]
        for wid in well_ids:
            p=data_dir/f'{wid}__horizontal_well.csv'
            try: df=pd.read_csv(p,usecols=['X','Y','ANCC']).dropna()
            except: continue
            if len(df)==0: continue
            ix=np.linspace(0,len(df)-1,min(spw,len(df)),dtype=int); s=df.iloc[ix]
            xs.append(s['X'].values); ys.append(s['Y'].values)
            anccs.append(s['ANCC'].values); wids.extend([wid]*len(s))
        self.xy=np.column_stack([np.concatenate(xs),np.concatenate(ys)])
        self.ancc=np.concatenate(anccs).astype(np.float32); self.wids=np.array(wids)
        self.scale=np.where(self.xy.std(0)<1e-3,1.,self.xy.std(0))
        self.tree=cKDTree(self.xy/self.scale)

    def impute(self,xy_q,self_wid=None,k=DENSE_K,nfetch=5000):
        xy_q=np.atleast_2d(xy_q); q=xy_q/self.scale; nf=min(nfetch,len(self.ancc))
        dist,idx=self.tree.query(q,k=nf,workers=-1)
        if self_wid: dist=np.where(self.wids[idx]==self_wid,np.inf,dist)
        ord=np.argpartition(dist,min(k-1,nf-1),1)[:,:k]
        dk=np.take_along_axis(dist,ord,1); ik=np.take_along_axis(idx,ord,1)
        vk=np.isfinite(dk); w=np.where(vk,1./(dk+1e-3),0.)
        sw=w.sum(1); safe=np.where(sw<1e-9,1.,sw); an=self.ancc[ik]
        ap=(an*w).sum(1)/safe; ap=np.where(sw<1e-9,float(self.ancc.mean()),ap)
        var=((an-ap[:,None])**2*w).sum(1)/safe
        return ap.astype(np.float32),np.sqrt(np.maximum(var,0.)).astype(np.float32),np.where(vk,dk,np.inf).min(1).astype(np.float32)

hw_paths=sorted((CFG.dataset_path / "train").glob('*__horizontal_well.csv'))
train_wids=[p.stem.replace('__horizontal_well','') for p in hw_paths]
FI=FormationPlaneKNN(train_wids,CFG.dataset_path / "train")
DI=DenseANCCImputer(train_wids,CFG.dataset_path / "train")
_FI=FI; _DI=DI

ANCH_OFFS=np.array([-80,-40,-20,-10,-5,0,5,10,20,40,80],np.float32)
BEAM_OFFS=np.array([-40,-20,-10,-5,-3,0,3,5,10,20,40],np.float32)
SC_OFFS  =np.array([-30,-15,-8,-4,-2,0,2,4,8,15,30],np.float32)
PF_OFFS  =np.array([-30,-15,-8,-4,-2,0,2,4,8,15,30],np.float32)
print('Spatial classes built:', len(train_wids), 'wells')


In [ ]:
def build_well(hw_path,tw_path,is_train):
    global _FI,_DI
    wid=Path(hw_path).stem.replace('__horizontal_well','')
    try:
        hw=pd.read_csv(hw_path); tw=pd.read_csv(tw_path).sort_values('TVT')
    except: return None
    if is_train and 'TVT' not in hw.columns: return None
    kn=hw[hw['TVT_input'].notna()]; ev=hw[hw['TVT_input'].isna()]
    if len(ev)==0 or len(kn)<10: return None
    if is_train and hw['TVT'].isna().all(): return None
    tw_tvt=tw['TVT'].to_numpy(np.float32); tw_gr=tw['GR'].to_numpy(np.float32)
    if len(tw_tvt)<3: return None

    pf_a,std_a=run_pf_ancc(hw,tw_tvt,tw_gr)
    if len(pf_a)==0: return None
    pf_z,std_z=run_pf_z(hw,tw_tvt,tw_gr)
    pf_use=pf_a.astype(np.float32); std_use=std_a.astype(np.float32)
    has_z=len(pf_z)==len(pf_a) and not np.any(np.isnan(pf_z))

    lk=kn.iloc[-1]; last_tvt=float(lk['TVT_input'])
    gr_full=hw['GR'].astype(float).interpolate(limit_direction='both').fillna(float(np.nanmean(tw_gr)))
    hgr=gr_full.iloc[ev.index[0]:].to_numpy(np.float32)
    kgr=gr_full.iloc[:len(kn)].to_numpy(np.float32)

    bpaths={}
    for (bs,mc,es,r,tag) in BEAMS:
        bpaths[tag]=beam_search(hgr,tw_tvt,tw_gr,last_tvt,bs,mc,es,r)
    beam_ref=(bpaths['cons']+bpaths['sm5'])/2.

    ktvt=kn['TVT_input'].to_numpy(np.float32)
    sc_res,sc_ens=multi_scale_ncc(kgr,ktvt,hgr,hws=(8,15,25,50),stride=3)
    sc8,sc8s=sc_res[0]; sc15,sc15s=sc_res[1]; sc25,sc25s=sc_res[2]
    sc50,sc50s=sc_res[3]
    sc_cons=(sc8+sc15+sc25)/3.
    sc_trust=float(np.clip(len(kn)/200.,0.,0.6))
    hyb_ref=(1-sc_trust)*beam_ref+sc_trust*sc_ens

    tw_at_k=np.interp(ktvt,tw_tvt,tw_gr).astype(np.float32)
    a_cal,b_cal=affine_cal(kgr,tw_at_k)
    kmd=kn['MD'].to_numpy(np.float32); kz=kn['Z'].to_numpy(np.float32)
    pfx_rmse=float(np.sqrt(np.mean((kgr-tw_at_k)**2)))
    slp_all=robust_slope(kmd,ktvt); slp_50=robust_slope(kmd[-50:],ktvt[-50:])
    slp_z=robust_slope(kz,ktvt)

    swid=wid if is_train else None
    xy_ev=ev[['X','Y']].to_numpy(np.float64); xy_kn=kn[['X','Y']].to_numpy(np.float64)
    form_ev,knn_d=_FI.impute(xy_ev,self_wid=swid)
    form_kn,_   =_FI.impute(xy_kn,self_wid=swid)
    z_kn=kn['Z'].to_numpy(np.float32); z_ev=ev['Z'].to_numpy(np.float32)

    tvt_fs={}; form_rmse={}; form_list=[]
    for fi2,fn in enumerate(FORMATIONS):
        b_full,b_early,b_mid,b_late,b_wls,b_trend=seg_b_well(ktvt,z_kn,form_kn[:,fi2])
        tvt_f  =(-z_ev+form_ev[:,fi2]+b_full ).astype(np.float32)
        tvt_fw =(-z_ev+form_ev[:,fi2]+b_wls  ).astype(np.float32)
        tvt_f50=(-z_ev+form_ev[:,fi2]+b_late ).astype(np.float32)
        tvt_ft =(-z_ev+form_ev[:,fi2]+b_trend).astype(np.float32)
        tvt_fs[f'tvtF_{fn}']=tvt_f; tvt_fs[f'tvtFw_{fn}']=tvt_fw
        tvt_fs[f'tvtF50_{fn}']=tvt_f50; tvt_fs[f'tvtFt_{fn}']=tvt_ft
        tvt_fs[f'bw_{fn}']=np.float32(b_full); tvt_fs[f'bww_{fn}']=np.float32(b_wls)
        tvt_fs[f'bw50_{fn}']=np.float32(b_late); tvt_fs[f'bwt_{fn}']=np.float32(b_trend)
        tvt_fs[f'bw_early_{fn}']=np.float32(b_early); tvt_fs[f'bw_mid_{fn}']=np.float32(b_mid)
        form_rmse[fn]=float(np.sqrt(np.mean((ktvt-(-z_kn+form_kn[:,fi2]+b_full))**2)))
        form_list.append(tvt_f)

    fs=np.stack(form_list,1)
    form_mean_d=(fs.mean(1)-last_tvt).astype(np.float32)
    form_std_d =fs.std(1).astype(np.float32)
    form_rng_d =(fs.max(1)-fs.min(1)).astype(np.float32)

    d_ancc,d_std,d_dist=_DI.impute(xy_ev,self_wid=swid)
    d_kn,d_std_kn,_=_DI.impute(xy_kn,self_wid=swid)
    b_vd=ktvt+z_kn-d_kn
    _,b_de,b_dm,b_dl,b_dw,b_dt=seg_b_well(ktvt,z_kn,d_kn)
    b_d=float(np.median(b_vd))
    tvt_dense  =(-z_ev+d_ancc+b_d  ).astype(np.float32)
    tvt_densew =(-z_ev+d_ancc+b_dw ).astype(np.float32)
    tvt_dense50=(-z_ev+d_ancc+b_dl ).astype(np.float32)
    tvt_denset =(-z_ev+d_ancc+b_dt ).astype(np.float32)
    res_kn=ktvt+z_kn-d_kn; d_rmse=float(np.sqrt(np.mean(res_kn**2)))
    d_bias=float(np.mean(res_kn)); d_nb_std=float(np.mean(d_std_kn))

    all_sigs=[pf_use]+[p for p in bpaths.values()]+[sc8,sc15,sc25,sc50,sc_ens,tvt_fs['tvtF_ANCC'],tvt_dense]
    sig_mat=np.stack(all_sigs,1)
    sig_std=sig_mat.std(1).astype(np.float32)
    sig_mean=(sig_mat.mean(1)-last_tvt).astype(np.float32)

    gr_s=pd.Series(gr_full.values); rolls={}
    for w in [3,5,11,21,51,101]:
        r=gr_s.rolling(w,center=True,min_periods=1)
        rolls[f'grm{w}']=r.mean().iloc[ev.index].values.astype(np.float32)
        rolls[f'grs{w}']=r.std().fillna(0).iloc[ev.index].values.astype(np.float32)
    for lag in [1,3,5,10,15,30]:
        rolls[f'glag{lag}']=gr_s.shift(lag).bfill().iloc[ev.index].values.astype(np.float32)
        rolls[f'glead{lag}']=gr_s.shift(-lag).ffill().iloc[ev.index].values.astype(np.float32)
    gr_d1=gr_s.diff().fillna(0.).iloc[ev.index].values.astype(np.float32)
    gr_d2=gr_s.diff().diff().fillna(0.).iloc[ev.index].values.astype(np.float32)
    gr_d3=gr_s.diff().diff().diff().fillna(0.).iloc[ev.index].values.astype(np.float32)
    gr_env=gr_s.rolling(21,center=True,min_periods=1).max().iloc[ev.index].values.astype(np.float32)
    gr_nrg=np.sqrt(np.maximum((gr_s**2).rolling(21,center=True,min_periods=1).mean(),0.)).iloc[ev.index].values.astype(np.float32)

    hmd=ev['MD'].to_numpy(np.float32); md_since=hmd-float(lk['MD'])
    slp_b_all=(last_tvt+slp_all*md_since).astype(np.float32)
    slp_b_50 =(last_tvt+slp_50 *md_since).astype(np.float32)

    mdd=hw['MD'].diff().replace(0,np.nan)
    dzdmd=(hw['Z'].diff()/mdd).iloc[ev.index].values.astype(np.float32)
    dxdmd=(hw['X'].diff()/mdd).iloc[ev.index].values.astype(np.float32)
    dydmd=(hw['Y'].diff()/mdd).iloc[ev.index].values.astype(np.float32)

    nh=len(ev); frac=(np.arange(nh)/max(nh-1,1)).astype(np.float32)
    def sc(v): return np.full(nh,np.float32(v),np.float32)

    feats={'well':wid,'id':[f'{wid}_{i}' for i in ev.index],
        'last_known_tvt':sc(last_tvt),
        'pf_ancc':pf_use,'pf_ancc_std':std_use,
        'pf_ancc_delta':(pf_use-last_tvt).astype(np.float32),
        'pf_z':(pf_z.astype(np.float32) if has_z else sc(last_tvt)),
        'pf_z_delta':((pf_z-last_tvt).astype(np.float32) if has_z else sc(0.)),
        'pf_vs_z':((pf_use-pf_z.astype(np.float32)) if has_z else sc(0.)),
        **{f'beam_{t}_d':(p-np.float32(last_tvt)).astype(np.float32) for t,p in bpaths.items()},
        'beam_mean_d':np.stack([(p-last_tvt) for p in bpaths.values()],1).mean(1).astype(np.float32),
        'beam_std_d':np.stack([(p-last_tvt) for p in bpaths.values()],1).std(1).astype(np.float32),
        'beam_med_d':np.median(np.stack([(p-last_tvt) for p in bpaths.values()],1),1).astype(np.float32),
        'beam_min_d':np.stack([(p-last_tvt) for p in bpaths.values()],1).min(1).astype(np.float32),
        'beam_max_d':np.stack([(p-last_tvt) for p in bpaths.values()],1).max(1).astype(np.float32),
        'sc8_d':(sc8-np.float32(last_tvt)).astype(np.float32),'sc8_sc':sc8s,
        'sc15_d':(sc15-np.float32(last_tvt)).astype(np.float32),'sc15_sc':sc15s,
        'sc25_d':(sc25-np.float32(last_tvt)).astype(np.float32),'sc25_sc':sc25s,
        'sc50_d':(sc50-np.float32(last_tvt)).astype(np.float32),'sc50_sc':sc50s,
        'sc_cons_d':(sc_cons-np.float32(last_tvt)).astype(np.float32),
        'sc_ens_d':(sc_ens-np.float32(last_tvt)).astype(np.float32),
        'sc_trust':sc(sc_trust),'hyb_d':(hyb_ref-np.float32(last_tvt)).astype(np.float32),
        'sig_std':sig_std,'sig_mean_d':sig_mean, **tvt_fs,
        **{f'frm_rmse_{fn}':sc(form_rmse[fn]) for fn in FORMATIONS},
        'form_mean_d':form_mean_d,'form_std_d':form_std_d,'form_rng_d':form_rng_d,
        'spatial_ancc_d':(form_ev[:,0]-np.float32(np.interp(last_tvt,tw_tvt,tw_gr))),
        'spatial_knn_dist':knn_d,'dense_ancc':d_ancc,'dense_std':d_std,'dense_dist':d_dist,
        'tvt_dense_d':(tvt_dense-last_tvt).astype(np.float32),
        'tvt_densew_d':(tvt_densew-last_tvt).astype(np.float32),
        'tvt_dense50_d':(tvt_dense50-last_tvt).astype(np.float32),
        'tvt_denset_d':(tvt_denset-last_tvt).astype(np.float32),
        'dense_rmse':sc(d_rmse),'dense_bias':sc(d_bias),'dense_nb_std':sc(d_nb_std),
        'pf_vs_spatial':(pf_use-tvt_fs['tvtF_ANCC']).astype(np.float32),
        'pf_vs_dense':(pf_use-tvt_dense).astype(np.float32),
        'spatial_vs_dense':(tvt_fs['tvtF_ANCC']-tvt_dense).astype(np.float32),
        'beam_vs_spatial':(bpaths['cons']-tvt_fs['tvtF_ANCC']).astype(np.float32),
        'sc_vs_beam':(sc_ens-bpaths['cons']).astype(np.float32),
        'cal_a':sc(a_cal),'cal_b':sc(b_cal),
        'pfx_rmse':sc(pfx_rmse),'known_len':sc(len(kn)),'eval_len':sc(nh),
        'slp_all':sc(slp_all),'slp_50':sc(slp_50),'slp_z':sc(slp_z),
        'slp_b_d_all':(slp_b_all-last_tvt).astype(np.float32),
        'slp_b_d_50':(slp_b_50-last_tvt).astype(np.float32),
        'ktvt_range':sc(float(np.ptp(ktvt))),'ktvt_std':sc(float(ktvt.std())),
        'ktvt_skew':sc(float(pd.Series(ktvt).skew() if len(ktvt)>2 else 0)),
        'ktvt_kurt':sc(float(pd.Series(ktvt).kurtosis() if len(ktvt)>3 else 0)),
        'md_since':md_since,'frac':frac,'frac2':frac**2,'sqrt_frac':np.sqrt(frac),
        'z':z_ev,'dx':(ev['X']-float(lk['X'])).to_numpy(np.float32),
        'dy':(ev['Y']-float(lk['Y'])).to_numpy(np.float32),
        'dz':(z_ev-float(lk['Z'])).astype(np.float32),
        'dxy':np.sqrt((ev['X']-float(lk['X']))**2+(ev['Y']-float(lk['Y']))**2).to_numpy(np.float32),
        'dzdmd':dzdmd,'dxdmd':dxdmd,'dydmd':dydmd,
        'gr':hgr,'gr_d1':gr_d1,'gr_d2':gr_d2,'gr_d3':gr_d3,'gr_env':gr_env,'gr_nrg':gr_nrg,
        'gr_vs_tw_anc':hgr-np.float32(np.interp(last_tvt,tw_tvt,tw_gr)),
        'gr_vs_slp_all':hgr-np.interp(slp_b_all,tw_tvt,tw_gr).astype(np.float32),
        **{f'tda{int(o)}':hgr-np.float32(np.interp(last_tvt+o,tw_tvt,tw_gr)) for o in ANCH_OFFS},
        **{f'tdbc{int(o)}':hgr-np.interp(beam_ref+o,tw_tvt,tw_gr).astype(np.float32) for o in BEAM_OFFS},
        **{f'tdsc{int(o)}':hgr-np.interp(sc_ens+o,tw_tvt,tw_gr).astype(np.float32) for o in SC_OFFS},
        **{f'tdpf{int(o)}':hgr-np.interp(pf_use+o,tw_tvt,tw_gr).astype(np.float32) for o in PF_OFFS},
        'tw_range':sc(float(np.ptp(tw_tvt))),'tw_gr_mean':sc(float(tw_gr.mean())),
        'tw_gr_std':sc(float(tw_gr.std())),'tw_gr_skew':sc(float(pd.Series(tw_gr).skew())),
    }
    for k,v in rolls.items(): feats[k]=v
    result=pd.DataFrame(feats)
    if is_train:
        if 'TVT' not in ev.columns or ev['TVT'].isna().all(): return None
        result['target']=(ev['TVT'].to_numpy(np.float32)-np.float32(last_tvt))
    return result

# Dataset builder
def build_dataset(paths,is_train,label):
    args=[(str(p),str(p.parent/f'{p.stem.replace("__horizontal_well","")}__typewell.csv'),is_train)
          for p in paths
          if (p.parent/f'{p.stem.replace("__horizontal_well","")}__typewell.csv').exists()]
    t0=time.time()
    res=Parallel(n_jobs=NCPU,prefer='threads',verbose=3)(
        delayed(build_well)(hp,tp,it) for hp,tp,it in args)
    parts=[r for r in res if r is not None]
    elapsed=time.time()-t0
    print(f'{label} dataset: {len(parts)} wells in {elapsed:.0f}s')
    return pd.concat(parts,ignore_index=True) if parts else pd.DataFrame()

print('build_well + build_dataset defined')


In [ ]:
if (CFG.artifacts_path / "data" / "train.csv").exists():
    train_df = pd.read_csv(CFG.artifacts_path / "data" / "train.csv", low_memory=False)
else:
    train_paths = sorted((CFG.dataset_path / "train").glob('*__horizontal_well.csv'))
    train_df = build_dataset(train_paths, is_train=True, label="train")

test_paths = sorted((CFG.dataset_path / "test").glob('*__horizontal_well.csv'))
test_df = build_dataset(test_paths, is_train=False, label="test")

features = [c for c in train_df.columns if c not in {'well','id','target'}]
X = train_df[features]
y = train_df['target']
g = train_df['well']
X_test = test_df[features]

print(f'Train: {train_df.shape}, Test: {test_df.shape}, Features: {len(features)}')
# Show data preview
train_df.head(3)


In [ ]:
device_type = "gpu" if CFG.USE_GPU else "cpu"
cb_device = "GPU" if CFG.USE_GPU else "CPU"

# === 4 LGBM variants ===
lgb_params = [
    dict(boosting_type="gbdt", num_leaves=255, min_child_samples=15,
         subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
         reg_lambda=3.0, reg_alpha=0.05, objective="regression",
         verbose=-1, n_jobs=-1, device_type=device_type, gpu_use_dp=False,
         max_bin=255, learning_rate=0.030, n_estimators=5000, seed=123),
    dict(n_jobs=-1, verbose=-1, reg_alpha=10.79, subsample=0.47,
         num_leaves=64, reg_lambda=95.75, n_estimators=10000,
         random_state=0, boosting_type='gbdt', learning_rate=0.00934,
         colsample_bytree=0.393, min_child_weight=0.24, min_child_samples=40,
         device_type=device_type),
    dict(n_jobs=-1, verbose=-1, reg_alpha=10.79, subsample=0.47,
         num_leaves=64, reg_lambda=95.75, n_estimators=10000,
         random_state=29, boosting_type='gbdt', learning_rate=0.00934,
         colsample_bytree=0.393, min_child_weight=0.24, min_child_samples=40,
         device_type=device_type),
    # NEW: 4th LGBM — more leaves, stronger regularization
    dict(n_jobs=-1, verbose=-1, reg_alpha=5.0, subsample=0.6,
         num_leaves=128, reg_lambda=50.0, n_estimators=8000,
         random_state=777, boosting_type='gbdt', learning_rate=0.015,
         colsample_bytree=0.5, min_child_samples=25,
         device_type=device_type),
]

# === 3 CatBoost variants ===
cb_params = [
    dict(iterations=8000, depth=7, l2_leaf_reg=2.0,
         min_data_in_leaf=15, border_count=254,
         loss_function="RMSE", task_type=cb_device, devices="0",
         od_type="Iter", od_wait=300, verbose=0,
         learning_rate=0.020, random_seed=7),
    dict(iterations=8000, depth=7, l2_leaf_reg=2.0,
         min_data_in_leaf=15, border_count=254,
         loss_function="RMSE", task_type=cb_device, devices="0",
         od_type="Iter", od_wait=300, verbose=0,
         learning_rate=0.030, random_seed=123),
    # NEW: deeper, slower learning, more iterations
    dict(iterations=10000, depth=8, l2_leaf_reg=3.0,
         min_data_in_leaf=10, border_count=254,
         loss_function="RMSE", task_type=cb_device, devices="0",
         od_type="Iter", od_wait=400, verbose=0,
         learning_rate=0.015, random_seed=42),
]

# === 3 XGBoost variants (NEW — not in original kernel) ===
xgb_params = [
    dict(n_estimators=5000, max_depth=8, learning_rate=0.03,
         subsample=0.8, colsample_bytree=0.8, reg_lambda=3.0,
         reg_alpha=0.05, min_child_weight=1, tree_method='hist',
         device="cuda" if CFG.USE_GPU else "cpu", random_state=123, verbosity=0),
    dict(n_estimators=8000, max_depth=6, learning_rate=0.02,
         subsample=0.7, colsample_bytree=0.7, reg_lambda=10.0,
         reg_alpha=1.0, min_child_weight=5, tree_method='hist',
         device="cuda" if CFG.USE_GPU else "cpu", random_state=456, verbosity=0),
    dict(n_estimators=10000, max_depth=5, learning_rate=0.015,
         subsample=0.6, colsample_bytree=0.6, reg_lambda=50.0,
         reg_alpha=5.0, min_child_weight=3, tree_method='hist',
         device="cuda" if CFG.USE_GPU else "cpu", random_state=789, verbosity=0),
]

# Ridge + ElasticNet stacking params
ridge_params = {"random_state": 42, "alpha": 1.66, "positive": False, "fit_intercept": True}
elas_params = {"random_state": 42, "alpha": 0.01, "l1_ratio": 0.5}

# Post-processing params
pp_params = {'alpha': 1.0, 'tau': 85.0, 'w_pf': 0.09}

print(f'Device: {device_type} | 10 models total (4 LGBM + 3 CB + 3 XGB)')


In [ ]:
oof_preds = {}
test_preds = {}
overall_scores = {}
fold_scores = {}


In [ ]:
for i, params in enumerate(lgb_params):   
    name = f"lightgbm-{i+1}"
    save_path = f"models/lightgbm-{i+1}"
    
    if (CFG.artifacts_path / save_path).exists():
        print(f"Loading lightgbm-{i+1} from disk...")
        trainer_paths = (CFG.artifacts_path / save_path).glob('*.pkl')
        trainer = joblib.load(list(trainer_paths)[0])
        print(f"Loaded lightgbm-{i+1} with OOF RMSE: {trainer.overall_score:.4f}")
    else:
        trainer = Trainer(
            estimator=LGBMRegressor(**params),
            task="regression",
            metric=CFG.metric,
            cv=CFG.cv,
            cv_args={"groups": g},
            use_early_stopping=True,
            verbose=True,
            save=True,
            save_path=save_path)
        trainer.fit(
            X, y,
            fit_args={"eval_metric": "rmse",
                     "callbacks": [log_evaluation(period=250), early_stopping(stopping_rounds=250)]})
        print(f"  {name} trained: {trainer.overall_score:.4f}")

    oof_preds[name] = trainer.oof_preds
    test_preds[name] = trainer.predict(X_test)
    overall_scores[name] = trainer.overall_score
    fold_scores[name] = trainer.fold_scores
    gc.collect()


In [ ]:
for i, params in enumerate(cb_params):    
    name = f"catboost-{i+1}"
    save_path = f"models/catboost-{i+1}"
    if (CFG.artifacts_path / save_path).exists():
        print(f"Loading catboost-{i+1} from disk...")
        trainer_paths = (CFG.artifacts_path / save_path).glob('*.pkl')
        trainer = joblib.load(list(trainer_paths)[0])
        print(f"Loaded catboost-{i+1} with OOF RMSE: {trainer.overall_score:.4f}")
    else:
        trainer = Trainer(
            estimator=CatBoostRegressor(**params),
            task="regression",
            metric=CFG.metric,
            cv=CFG.cv,
            cv_args={"groups": g},
            use_early_stopping=True,
            verbose=True,
            save=True,
            save_path=save_path)
        trainer.fit(X, y,
            fit_args={"verbose": 250, "early_stopping_rounds": 250, "use_best_model": True})
        print(f"  {name} trained: {trainer.overall_score:.4f}")

    oof_preds[name] = trainer.oof_preds
    test_preds[name] = trainer.predict(X_test)
    overall_scores[name] = trainer.overall_score
    fold_scores[name] = trainer.fold_scores
    gc.collect()


In [ ]:
# === NEW: 3 XGBoost models (not in original kernel) ===
for i, params in enumerate(xgb_params):
    name = f"xgboost-{i+1}"
    oof = np.zeros(len(X)); test_pred = np.zeros(len(X_test))
    fold_scores_list = []
    for fold, (tr_idx, va_idx) in enumerate(CFG.cv.split(X, y, g)):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
        model = XGBRegressor(**params)
        model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
        oof[va_idx] = model.predict(X_va)
        test_pred += model.predict(X_test) / CFG.n_splits
        fold_scores_list.append(root_mean_squared_error(y_va, oof[va_idx]))
    oof_preds[name] = oof
    test_preds[name] = test_pred
    overall_scores[name] = float(np.mean(fold_scores_list))
    fold_scores[name] = fold_scores_list
    print(f"  {name} OOF RMSE: {overall_scores[name]:.4f}")


In [ ]:
oof_preds = pd.DataFrame(oof_preds)
test_preds = pd.DataFrame(test_preds)
print(f'OOF shape: {oof_preds.shape}, Test preds shape: {test_preds.shape}')


In [ ]:
# --- Ridge stacking ---
ridge_trainer = Trainer(
    Ridge(**ridge_params),
    task="regression",
    metric=CFG.metric,
    cv=CFG.cv,
    cv_args={"groups": g},
    verbose=True)

ridge_trainer.fit(oof_preds, y)
ridge_oof = ridge_trainer.oof_preds
ridge_test = ridge_trainer.predict(test_preds)
overall_scores["ridge"] = ridge_trainer.overall_score
fold_scores["ridge"] = ridge_trainer.fold_scores
print(f'Ridge stack OOF: {ridge_trainer.overall_score:.4f}')

# --- ElasticNet stacking (robustness) ---
class ElasticNetTrainer:
    def __init__(self, estimator, cv, cv_args):
        self.estimator=estimator; self.cv=cv; self.cv_args=cv_args
    def fit(self, X, y):
        self.oof_preds=np.zeros(len(X))
        fold_scores=[]
        for tr,va in self.cv.split(X,y,**self.cv_args):
            self.estimator.fit(X.iloc[tr],y.iloc[tr])
            self.oof_preds[va]=self.estimator.predict(X.iloc[va])
            fold_scores.append(root_mean_squared_error(y.iloc[va],self.oof_preds[va]))
        self.overall_score=float(np.mean(fold_scores))
    def predict(self, X): return self.estimator.predict(X)

elas_trainer = ElasticNetTrainer(ElasticNet(**elas_params), CFG.cv, {"groups":g})
elas_trainer.fit(oof_preds, y)
elas_oof = elas_trainer.oof_preds
elas_test = elas_trainer.predict(test_preds)
overall_scores["elasticnet"] = elas_trainer.overall_score
print(f'ElasticNet stack OOF: {elas_trainer.overall_score:.4f}')

# --- Average both stacks ---
stack_oof = (ridge_oof + elas_oof) / 2.
stack_test = (ridge_test + elas_test) / 2.
overall_scores["stack_avg"] = root_mean_squared_error(y, stack_oof)
print(f'Average Stack OOF: {overall_scores["stack_avg"]:.4f}')


In [ ]:
def apply_pp(df, md, pd_, alpha=1.0, tau=85, w_pf=0.09):
    d = md * (1-w_pf) + pd_ * w_pf
    if tau: 
        d *= (1.-np.exp(-np.maximum(df['md_since'].values,0.) / tau))
    return d * alpha

def sg_smooth(df, col, sg_w=17, sg_p=3):
    df = df.copy()
    for _, g in df.groupby('well', sort=False):
        v = g[col].values; n = len(v)
        wl = min(sg_w, n)
        if wl % 2 == 0: wl -= 1
        if wl >= sg_p + 2:
            v = savgol_filter(v, wl, sg_p)
        df.loc[g.index, col] = v
    return df

def robust_poly_fit(df, col='tvt', deg=4, blend=0.75):
    out = {}
    for wid, g in df.groupby('well', sort=False):
        try:
            g = g.sort_values('md_since')
            md = g['md_since'].values.astype(float)
            Z = g['z'].values.astype(float)
            tvt = g[col].values.astype(float)
            last_tvt = float(g['last_known_tvt'].iloc[0])
            anchor = last_tvt + Z[0]
            y = tvt + Z - anchor
            s = md / max(md[-1], 1e-6)
            if len(s) < deg + 2:
                for i, rid in enumerate(g['id'].values):
                    out[rid] = float(tvt[i])
                continue
            c = np.polyfit(s, y, deg)
            for _ in range(5):
                r = y - np.polyval(c, s)
                sc = np.median(np.abs(r)) * 1.4826 + 1e-6
                c = np.polyfit(s, y, deg, w=1.0 / (1.0 + (r / (2.0 * sc)) ** 2))
            y_fit = np.polyval(c, s)
            tvt_fit_full = (anchor + y_fit) - Z
            tvt_fit = blend * tvt_fit_full + (1 - blend) * tvt
            for i, rid in enumerate(g['id'].values):
                out[rid] = float(tvt_fit[i])
        except:
            for i, rid in enumerate(g['id'].values):
                out[rid] = float(g[col].values[i]) if rid not in out else out[rid]
    return out

print('Post-processing functions defined')


In [ ]:
base_tvt = train_df['last_known_tvt'].values
y_true_abs = y.values + base_tvt
pf_oof = (train_df['pf_ancc'].values - base_tvt)

d = apply_pp(train_df, stack_oof, pf_oof, **pp_params)
stack_pp_score = root_mean_squared_error(y_true_abs, base_tvt + d)
overall_scores["stack_pp"] = stack_pp_score
print(f'Stack+PP OOF (default): {stack_pp_score:.4f}')

# === Grid search best PP params ===
best_pp_score = stack_pp_score
best_pp_params = pp_params.copy()
for alpha in [0.8, 0.9, 1.0, 1.1, 1.2]:
    for tau in [60, 75, 85, 100, 120]:
        for w_pf in [0.05, 0.07, 0.09, 0.12, 0.15]:
            d = apply_pp(train_df, stack_oof, pf_oof, alpha=alpha, tau=tau, w_pf=w_pf)
            s = root_mean_squared_error(y_true_abs, base_tvt + d)
            if s < best_pp_score:
                best_pp_score = s
                best_pp_params = {'alpha': alpha, 'tau': tau, 'w_pf': w_pf}
print(f'Best Stack+PP OOF: {best_pp_score:.4f} @ {best_pp_params}')


In [ ]:
test_df2 = test_df.copy()
pf_test = test_df2['pf_ancc'].values - test_df2['last_known_tvt'].values

test_df2['pred'] = test_df2['last_known_tvt'].values + apply_pp(
    test_df2, stack_test, pf_test, **best_pp_params)
test_df2 = sg_smooth(test_df2, 'pred')

# Apply robust polynomial projection to Engine 1
poly_eng1 = robust_poly_fit(test_df2, col='pred', deg=4, blend=0.75)
test_df2['pred_final'] = test_df2['id'].map(poly_eng1).fillna(test_df2['pred'])

sample_sub = pd.read_csv(CFG.dataset_path / "sample_submission.csv")
sub_1 = (sample_sub[['id']].merge(
    test_df2[['id', 'pred_final']].rename(columns={'pred_final':'tvt'}),
    on='id', how='left'))

sub_1['tvt'] = sub_1['tvt'].fillna(float(train_df['last_known_tvt'].mean()+train_df['target'].mean()))
print(f'Engine 1 (ML) submission: {len(sub_1)} rows')
sub_1.head()


In [ ]:
sample = pd.read_csv(CFG.dataset_path / 'sample_submission.csv')
sample['well']    = sample['id'].str[:8]
sample['row_idx'] = sample['id'].str[9:].astype(int)

train_hw_files = sorted(glob.glob(str(CFG.dataset_path / 'train' / '*__horizontal_well.csv')))
train_wells = [os.path.basename(f).split('__')[0] for f in train_hw_files]

test_hw_files = sorted(glob.glob(str(CFG.dataset_path / 'test' / '*__horizontal_well.csv')))
test_wells = [os.path.basename(f).split('__')[0] for f in test_hw_files]

print(f'Train wells: {len(train_wells)}, Test wells: {len(test_wells)}')

rows = []
for i, wid in enumerate(test_wells):
    print(f'[Engine 2 {i+1}/{len(test_wells)}] {wid}...')
    hw_te = pd.read_csv(CFG.dataset_path / 'test' / f'{wid}__horizontal_well.csv')
    tw_te = pd.read_csv(CFG.dataset_path / 'test' / f'{wid}__typewell.csv').sort_values('TVT')

    # Overlap wells: use train copy's TVT_input
    if wid in train_wells:
        try:
            hw_tr = pd.read_csv(CFG.dataset_path / 'train' / f'{wid}__horizontal_well.csv')
            hw_te['TVT_input'] = hw_tr['TVT_input'].values
        except: pass

    selector_code, selector_variant, selector_n_eval, selector_z_span = selector_well_code(hw_te)

    # 128-seed likelihood-weighted PF
    try:
        tw_ref = tw_te
        pf_by_scale = run_pf_lik_ensemble_scales(hw_te, tw_ref, scales=SELECTOR_SCALES, n_particles=500, n_seeds=128)
        tvt_pf = pf_by_scale.get('pf_scale_8', pf_by_scale.get('pf_mean'))
        print(f'  PF 128-seed OK')
    except Exception as e:
        print(f'  PF failed: {e}')
        last_known = hw_te['TVT_input'].dropna()
        last_val = float(last_known.iloc[-1]) if len(last_known) > 0 else 0.0
        tvt_pf = hw_te['TVT_input'].fillna(last_val).values.astype(float)
        pf_by_scale = {f'pf_scale_{scale:g}': tvt_pf.copy() for scale in SELECTOR_SCALES}

    # Beam ensemble
    try:
        tvt_beam = run_beam_ensemble(hw_te, tw_ref)
        print(f'  Beam OK')
    except Exception as e:
        print(f'  Beam failed: {e}')
        tvt_beam = tvt_pf.copy()

    # Selector blending
    last_known = hw_te['TVT_input'].dropna()
    last_known_tvt = float(last_known.iloc[-1]) if len(last_known) > 0 else float(np.nanmean(tvt_pf))
    tvt_selector = apply_selector_variant(selector_variant, pf_by_scale, tvt_beam, last_known_tvt)

    ws = sample[sample['well'] == wid]
    for _, row in ws.iterrows():
        ridx = int(row['row_idx'])
        tvt_val = float(tvt_selector[ridx]) if ridx < len(tvt_selector) else last_known_tvt
        rows.append({'id': row['id'], 'tvt': tvt_val})
    print(f'  Added {len(ws)} rows')


In [ ]:
sub_2 = pd.DataFrame(rows)
print(f'Engine 2 (PF Selector) submission: {len(sub_2)} rows')
sub_2.head()


In [ ]:
sub = (
    sub_1.merge(sub_2, on='id', suffixes=('_1', '_2'))
)

# Adaptive blending: longer wells get more PF weight (better PF tracking), shorter ones more ML
sub['well'] = sub['id'].str[:8]
well_sizes = sub.groupby('well').size()
sub['n_rows'] = sub['well'].map(well_sizes)

# Per-well blend weight: n_rows > 5000 → 0.70 PF, 5000 ≥ n > 2000 → 0.60 PF, else → 0.55 PF
def blend_weight(n):
    if n > 5000: return 0.70
    elif n > 2000: return 0.60
    else: return 0.55

sub['w_pf'] = sub['n_rows'].apply(blend_weight)
sub['tvt'] = (1 - sub['w_pf']) * sub['tvt_1'] + sub['w_pf'] * sub['tvt_2']
sub = sub[['id', 'tvt']]

sub.to_csv("submission.csv", index=False)
print(f'Blended submission: {len(sub)} rows')
sub.head()


In [ ]:
# Runs AFTER the final blend writes submission.csv; OVERWRITES it with projected version.
import numpy as _np, pandas as _pd

def _robfit(s, y, deg=4):
    if len(s) < deg + 2: return y.copy()
    c = _np.polyfit(s, y, deg)
    for _ in range(5):
        r = y - _np.polyval(c, s)
        sc = _np.median(_np.abs(r)) * 1.4826 + 1e-6
        c = _np.polyfit(s, y, deg, w=1.0 / (1.0 + (r / (2.0 * sc)) ** 2))
    return _np.polyval(c, s)

try:
    _base = _pd.read_csv("submission.csv")
    assert set(['id','tvt']).issubset(_base.columns)
    _base['well'] = _base['id'].str[:8]
    _base['row_idx'] = _base['id'].str[9:].astype(int)
    _out = dict(zip(_base['id'].values, _base['tvt'].astype(float).values))
    _n_ok = 0
    for _wid, _g in _base.groupby('well'):
        try:
            _hw = _pd.read_csv(CFG.dataset_path / 'test' / (_wid + '__horizontal_well.csv'))
            _kn = _hw[_hw['TVT_input'].notna()]
            if len(_kn) < 5: continue
            _last = _kn.iloc[-1]
            _anchor = float(_last['TVT_input']) + float(_last['Z'])
            _ps = float(_last['MD']); _end = float(_hw['MD'].iloc[-1])
            _gi = _g.sort_values('row_idx')
            _ri = _gi['row_idx'].values
            _Z = _hw['Z'].values[_ri].astype(float)
            _md = _hw['MD'].values[_ri].astype(float)
            _s = (_md - _ps) / max(_end - _ps, 1e-6)
            _tvt = _gi['tvt'].values.astype(float)
            _fit = _robfit(_s, (_tvt + _Z) - _anchor, 4)
            _tvt_fit_full = (_anchor + _fit) - _Z
            _tvt_fit = 0.25 * _tvt + 0.75 * _tvt_fit_full
            if not _np.all(_np.isfinite(_tvt_fit)): continue
            for _rid, _val in zip(_gi['id'].values, _tvt_fit):
                _out[_rid] = float(_val)
            _n_ok += 1
        except Exception as _e:
            print('proj fallback', _wid, _e)
    print(f'Projection applied to {_n_ok} wells')
    _final = _base[['id']].copy()
    _final['tvt'] = _final['id'].map(_out).astype(float)
    _final[['id','tvt']].to_csv("submission.csv", index=False)
    print(f'Wrote projected submission.csv {_final.shape}')
except Exception as _e:
    print('PROJECTION SKIPPED (kept blended submission):', _e)


In [ ]:
fold_scores_df = pd.DataFrame(fold_scores)
overall_scores_df = pd.DataFrame({k: [v] for k, v in overall_scores.items()}).transpose().sort_values(by=0, ascending=True)
print('\n=== OOF Score Summary (lower = better) ===')
print(overall_scores_df.to_string())


In [ ]:
# === Gold visible-prefix calibration overlay ===
# Runs AFTER the submission.csv is written.
# Per well: tests candidate predictors against the visible prefix, picks the winner.
import os as _gold_os

_GOLD_ENABLE = _gold_os.environ.get('ROGII_GOLD_PREFIX_CAL', '1') == '1'
_GOLD_PROFILE = _gold_os.environ.get('ROGII_GOLD_PROFILE', 'balanced')

if _GOLD_ENABLE:
    print(f'Gold calibration enabled: profile={_GOLD_PROFILE}')
    
    # The full gold calibration is in the companion gold_calibration.py
    # For brevity in this notebook, we leverage the environment variable to signal
    # the external calibration script.
    
    _gold_sub = pd.read_csv('submission.csv')
    _gold_sample = pd.read_csv(CFG.dataset_path / 'sample_submission.csv')
    
    _gold_sub['well'] = _gold_sub['id'].str[:8]
    _gold_sub['row_idx'] = _gold_sub['id'].str[9:].astype(int)
    
    test_wells_gold = sorted(_gold_sub['well'].unique())
    gold_moves = 0
    out_gold = dict(zip(_gold_sub['id'].values, _gold_sub['tvt'].values))
    
    for wi, wid in enumerate(test_wells_gold):
        print(f'[gold {wi+1}/{len(test_wells_gold)}] {wid}')
        try:
            hw_g = pd.read_csv(CFG.dataset_path / 'test' / f'{wid}__horizontal_well.csv')
            tw_g = pd.read_csv(CFG.dataset_path / 'test' / f'{wid}__typewell.csv')
            
            g = _gold_sub[_gold_sub['well'] == wid]
            
            # Get the visible prefix (known TVT_input points)
            kn_g = hw_g[hw_g['TVT_input'].notna()]
            n_prefix = len(kn_g)
            
            if n_prefix < 140:
                print(f'  skip: short prefix ({n_prefix})')
                continue
            
            # Build a simple holdout: hide last 30% of prefix, test candidates
            cut = int(n_prefix * 0.70)
            hold_idx = kn_g.index[cut:]
            cutoff_idx = kn_g.index[cut - 1]
            
            if len(hold_idx) < 35:
                continue
            
            # Create truncated hw (hide holdout portion)
            hw_m = hw_g.copy(deep=True)
            hw_m.loc[hw_m.index > cutoff_idx, 'TVT_input'] = np.nan
            
            # Run select PF variants
            try:
                pf_candidates = run_pf_lik_ensemble_scales(hw_m, tw_g, scales=(3,5,8,12), n_particles=350, n_seeds=24)
                tvt_beam = run_beam_ensemble(hw_m, tw_g)
                
                best_name = None; best_rmse = float('inf')
                for vname in [
                    'pf_scale_5_hold_0.2', 'pf_scale_3_hold_0.15', 'pf_scale_8_hold_0.2',
                    'pf_scale_12_hold_0.15', 'pf_scale_5_hold_0.15']:
                    pred = apply_selector_variant(vname, pf_candidates, tvt_beam, float(kn_g.iloc[-1]['TVT_input']))
                    if len(pred) == len(hw_g):
                        err = root_mean_squared_error(hw_g.loc[hold_idx,'TVT'].values if 'TVT' in hw_g.columns else hw_g.loc[hold_idx,'TVT_input'].values, pred[hold_idx])
                        if err < best_rmse:
                            best_rmse = err
                            best_name = vname
                
                if best_name:
                    # Re-run winner with full prefix
                    pf_full = run_pf_lik_ensemble_scales(hw_g, tw_g, scales=SELECTOR_SCALES, n_particles=500, n_seeds=128)
                    tvt_beam_f = run_beam_ensemble(hw_g, tw_g)
                    winner = apply_selector_variant(best_name, pf_full, tvt_beam_f, float(kn_g.iloc[-1]['TVT_input']))
                    
                    ri = g['row_idx'].values
                    for j, rid in enumerate(g['id'].values):
                        if 0 <= ri[j] < len(winner) and np.isfinite(winner[ri[j]]):
                            out_gold[rid] = float(winner[ri[j]])
                    gold_moves += 1
                    print(f'  applied {best_name} (prefix RMSE={best_rmse:.3f})')
            except Exception as e:
                print(f'  gold skip: {e}')
        except Exception as e:
            print(f'  gold error: {e}')
    
    _gold_base = pd.read_csv('submission.csv')[['id']]
    _gold_base['tvt'] = _gold_base['id'].map(out_gold).astype(float)
    _gold_base[['id','tvt']].to_csv('submission.csv', index=False)
    print(f'Gold calibration: moved {gold_moves}/{len(test_wells_gold)} wells')
else:
    print('Gold calibration disabled (ROGII_GOLD_PREFIX_CAL=0)')


In [ ]:
from pathlib import Path as _BlendPath
import pandas as _blend_pd
_sp45_path = _BlendPath('/kaggle/working/submission.csv') if _BlendPath('/kaggle/working').exists() else _BlendPath('submission.csv')
_sp45_df = _blend_pd.read_csv(_sp45_path)
_sp45_df.to_csv((_BlendPath('/kaggle/working/submission.csv') if _BlendPath('/kaggle/working').exists() else _BlendPath('submission.csv')), index=False)
print(f'Final submission written: {len(_sp45_df)} rows')
print(f'Min tvt: {_sp45_df["tvt"].min():.1f}, Max tvt: {_sp45_df["tvt"].max():.1f}, Mean tvt: {_sp45_df["tvt"].mean():.1f}')
_sp45_df.head()
